# 🛰️ SongHong SAR Monitoring — Tuần 1
## Quy trình chuẩn bị dữ liệu, tiền xử lý và thiết lập môi trường GEE (Phiên bản Tối ưu)

**Người thực hiện:** Vũ Đức Tùng  
**Tháng thực hiện:** 07/2026  
**Đề tài:** Giám sát biến động đường bờ sông bãi bồi Sông Hồng tại Hà Nội bằng dữ liệu Sentinel-1 SAR  
**GEE Project ID:** `crested-library-500309-i2`  

---

### 🌟 Các cải tiến trong cấu trúc mô-đun mới:
1. **Mô-đun hóa (`src/`):** Tách cấu hình, tiền xử lý, thu thập dữ liệu và tiện ích thành các tệp Python sạch.
2. **Tự động hóa Asset:** Tự động đồng bộ hóa AOI GeoJSON lên GEE Asset qua Python API thay vì chạy CLI.
3. **Ép kiểu an toàn (Safe Casting):** Ép kiểu `ee.Image` rõ ràng tại các đầu ra của bộ lọc speckle để đảm bảo không bị lỗi trong Python.
4. **Kiểm chứng tự động:** Thực hiện so sánh giá trị backscatter thực tế của mặt nước và đất liền với các ngưỡng lý thuyết để xác thực chất lượng tiền xử lý.

## ⚡ Bước 1 — Thiết lập môi trường và nạp AOI

Khởi tạo kết nối GEE, tải vùng nghiên cứu (AOI) từ GeoJSON local và tự động kiểm tra/upload AOI lên GEE Assets của dự án.

In [1]:
import ee
import geemap
import os
import sys
from datetime import datetime

# Thêm thư mục gốc vào path để import src package
sys.path.insert(0, os.getcwd())

from src.config import GEE_PROJECT, AOI_GEOJSON_PATH, WATER_REF_POINT, LAND_REF_POINT
from src.aoi import get_aoi_collection, get_aoi_geometry, sync_aoi_to_assets
from src.collection import get_s1_collection, get_processed_collection, get_monthly_composite, get_annual_composite, get_coverage_statistics
from src.utils import save_interactive_map, save_stats_to_csv, save_stats_to_json, verify_backscatter_values

# Khởi tạo Google Earth Engine
ee.Initialize(project=GEE_PROJECT)
print(f"✅ GEE Initialized successfully with project: '{GEE_PROJECT}'")

# Tự động kiểm tra và đồng bộ hóa AOI lên GEE Assets
sync_aoi_to_assets()

# Load AOI
aoi_fc = get_aoi_collection()
aoi_geom = get_aoi_geometry()
print(f"✅ AOI loaded: {aoi_fc.size().getInfo()} features, area approx 500 km2")

C:\Users\Admin\AppData\Local\Programs\Python\Python310\lib\site-packages\google\api_core\_python_version_support.py:255: FutureWarning: You are using a Python version (3.10.10) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)


✅ GEE Initialized successfully with project: 'crested-library-500309-i2'


[AOI] Asset already exists at: projects/crested-library-500309-i2/assets/song_hong_aoi


[AOI] Loaded from GEE Asset: projects/crested-library-500309-i2/assets/song_hong_aoi


[AOI] Loaded from GEE Asset: projects/crested-library-500309-i2/assets/song_hong_aoi


✅ AOI loaded: 1 features, area approx 500 km2


## 🔍 Bước 2 — Kiểm tra môi trường & metadata Sentinel-1

Kiểm tra kết nối tới bộ dữ liệu Sentinel-1 GRD thô, in ra thông số metadata của ảnh đầu tiên thu được trong năm 2024 để xác minh độ phân giải và các polarization.

In [2]:
raw_col_2024 = get_s1_collection(aoi_geom, '2024-01-01', '2024-12-31')
count_2024 = raw_col_2024.size().getInfo()
print(f"📊 Số lượng ảnh Sentinel-1 thô trong AOI năm 2024: {count_2024} ảnh")

first_img = raw_col_2024.first()
first_info = first_img.getInfo()
print("\n📋 Metadata ảnh đầu tiên:")
print(f"   ID         : {first_info['id']}")
print(f"   Polarizations: {first_info['properties']['transmitterReceiverPolarisation']}")
print(f"   Resolution : {first_info['bands'][0].get('crs_transform', 'N/A')}")

📊 Số lượng ảnh Sentinel-1 thô trong AOI năm 2024: 31 ảnh



📋 Metadata ảnh đầu tiên:
   ID         : COPERNICUS/S1_GRD/S1A_IW_GRDH_1SDV_20240104T225116_20240104T225141_051963_06475D_B3A5
   Polarizations: ['VV', 'VH']
   Resolution : [10, 0, 446599.458095774, 0, -10, 2428034.9633712517]


## 📊 Bước 3 — Thu thập dữ liệu và Phân tích độ phủ (2015–2024)

Thống kê số lượng ảnh Sentinel-1 khả dụng trong AOI theo từng năm và tháng trong suốt chuỗi 10 năm (2015-2024) nhằm phát hiện các khoảng trống dữ liệu (gap). Báo cáo thống kê được tự động lưu dưới định dạng CSV và JSON.

In [3]:
raw_col_10yr = get_s1_collection(aoi_geom, '2015-01-01', '2024-12-31')
year_stats, month_stats = get_coverage_statistics(raw_col_10yr)

# In bảng thống kê theo năm
print("📅 Số ảnh theo năm (2015-2024):")
print(f"{'Năm':<8} | {'Số ảnh':<8} | {'Trạng thái'}")
print("-" * 32)
for stat in year_stats:
    print(f"{stat['year']:<8} | {stat['count']:<8} | {stat['status']}")

# Lưu báo cáo ra file CSV/JSON
save_stats_to_csv(year_stats, 's1_coverage_by_year.csv', ['year', 'count', 'status'])
save_stats_to_csv(month_stats, 's1_coverage_by_month.csv', ['month', 'month_name', 'count'])

report_summary = {
    'generated_at': datetime.utcnow().isoformat() + 'Z',
    'project': GEE_PROJECT,
    'total_images_10yr': raw_col_10yr.size().getInfo(),
    'year_stats': year_stats,
    'month_stats_2020_2024': month_stats
}
save_stats_to_json(report_summary, 's1_metadata.json')
print("\n✅ Báo cáo thống kê độ phủ đã được lưu thành công!")

📅 Số ảnh theo năm (2015-2024):
Năm      | Số ảnh   | Trạng thái
--------------------------------
2015     | 37       | ✅ OK
2016     | 37       | ✅ OK
2017     | 29       | ✅ OK
2018     | 31       | ✅ OK
2019     | 29       | ✅ OK
2020     | 34       | ✅ OK
2021     | 29       | ✅ OK
2022     | 30       | ✅ OK
2023     | 30       | ✅ OK
2024     | 31       | ✅ OK
[Stats] Saved CSV report to: D:\Future Career\SongHong-SAR-Monitoring\outputs\s1_coverage_by_year.csv
[Stats] Saved CSV report to: D:\Future Career\SongHong-SAR-Monitoring\outputs\s1_coverage_by_month.csv


[Stats] Saved JSON report to: D:\Future Career\SongHong-SAR-Monitoring\outputs\s1_metadata.json

✅ Báo cáo thống kê độ phủ đã được lưu thành công!


## ⚙️ Bước 4 — Tiền xử lý dữ liệu & Trích xuất đặc trưng

Thiết lập bộ sưu tập đã qua xử lý (`get_processed_collection`) bao gồm:
- Clip ảnh theo AOI.
- Lọc nhiễu speckle bằng bộ lọc Refined Lee (3x3 kernel) trên linear scale.
- Tính toán đặc trưng VV, VH và tỉ số VV/VH ratio (tương ứng với VV - VH trên dB scale).

Sau đó, tiến hành **kiểm chứng tự động (Validation Check)** chất lượng tiền xử lý bằng cách so sánh trị giá VV trung bình của hai điểm tham chiếu nước và đất với các ngưỡng vật lý lý thuyết.

In [4]:
processed_col = get_processed_collection(aoi_geom, '2015-01-01', '2024-12-31')
print(f"✅ Đã thiết lập pipeline tiền xử lý cho {processed_col.size().getInfo()} ảnh!")

# Kiểm chứng chất lượng lọc nhiễu speckle trên ảnh mùa khô (Tháng 1/2024)
jan_2024_comp = get_monthly_composite(processed_col, 2024, 1, aoi_geom)

# Thực hiện kiểm chứng tự động giá trị backscatter
validation_results = verify_backscatter_values(jan_2024_comp, WATER_REF_POINT, LAND_REF_POINT)
save_stats_to_json(validation_results, 'preprocessing_stats.json')
print("✅ Kết quả kiểm chứng đặc trưng đã được lưu vào outputs/preprocessing_stats.json")

✅ Đã thiết lập pipeline tiền xử lý cho 317 ảnh!



[Validation] Backscatter Validation Check:
   Water Point VV: -15.71 dB (Expected < -15.0 dB) -> PASSED
   Land Point VV : -2.71 dB (Expected > -10.0 dB) -> PASSED
[Stats] Saved JSON report to: D:\Future Career\SongHong-SAR-Monitoring\outputs\preprocessing_stats.json
✅ Kết quả kiểm chứng đặc trưng đã được lưu vào outputs/preprocessing_stats.json


## 🗺️ Bước 5 — Bản đồ tương tác & So sánh mùa khô/lũ

Tạo các bản đồ tương tác sử dụng thư viện `geemap` để so sánh trực quan:
1. **Biến động theo mùa:** So sánh ảnh composite mùa khô (Tháng 1/2020) và mùa lũ (Tháng 8/2020) để thấy rõ phần bãi bồi bị ngập nước.
2. **So sánh 3 đặc trưng:** Xem các đặc trưng VV, VH và VV/VH ratio cùng với ảnh Sentinel-2 RGB để đánh giá khả năng phân tách giữa nước, bãi cát và thực vật.

In [5]:
# 1. Tạo composite mùa khô (tháng 1) và mùa lũ (tháng 8)
dry_comp_2020 = get_monthly_composite(processed_col, 2020, 1, aoi_geom)
wet_comp_2020 = get_monthly_composite(processed_col, 2020, 8, aoi_geom)

# 2. Bản đồ 1: So sánh mùa khô vs mùa lũ (Sử dụng VV band)
Map1 = geemap.Map(center=[21.04, 105.86], zoom=11)
Map1.add_basemap('SATELLITE')
vis_vv = {'bands': ['VV'], 'min': -22, 'max': -2, 'palette': ['#000033', '#003399', '#0099FF', '#FFFFFF']}

Map1.addLayer(aoi_geom, {'color': 'red', 'width': 1.5}, 'AOI Boundary')
Map1.addLayer(dry_comp_2020, vis_vv, 'Sentinel-1 VV — Mùa khô (T1/2020)')
Map1.addLayer(wet_comp_2020, vis_vv, 'Sentinel-1 VV — Mùa lũ (T8/2020)')
save_interactive_map(Map1, 'visualization_dry_wet.html')

# 3. Bản đồ 2: So sánh đặc trưng và Sentinel-2 đối chiếu
Map2 = geemap.Map(center=[21.04, 105.86], zoom=11)
Map2.add_basemap('SATELLITE')

# Load Sentinel-2 RGB đối chiếu (Tháng 1/2020)
s2_ref = (ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
          .filterBounds(aoi_geom)
          .filterDate('2020-01-01', '2020-01-31')
          .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 20))
          .select(['B4', 'B3', 'B2'])
          .median()
          .clip(aoi_geom))

vis_ratio = {'bands': ['VV_VH_ratio'], 'min': 3, 'max': 12, 'palette': ['#440154', '#31688E', '#35B779', '#FDE725']}

Map2.addLayer(s2_ref, {'bands': ['B4', 'B3', 'B2'], 'min': 0, 'max': 3000}, 'Sentinel-2 RGB (T1/2020)')
Map2.addLayer(dry_comp_2020, vis_vv, 'Sentinel-1 VV (T1/2020)')
Map2.addLayer(dry_comp_2020, vis_ratio, 'Sentinel-1 VV/VH Ratio (T1/2020)')
save_interactive_map(Map2, 'visualization_features.html')

[Map] Saved interactive map to: D:\Future Career\SongHong-SAR-Monitoring\outputs\visualization_dry_wet.html


[Map] Saved interactive map to: D:\Future Career\SongHong-SAR-Monitoring\outputs\visualization_features.html


'D:\\Future Career\\SongHong-SAR-Monitoring\\outputs\\visualization_features.html'

## 📤 Bước 6 — Xuất dữ liệu mẫu lên Google Drive
Gửi các task xuất ảnh composite ra định dạng GeoTIFF độ phân giải 10m trên hệ tọa độ VN2000/UTM Zone 48N (`EPSG:32648`) lên GEE Server.

In [6]:
# Google Drive Export has been removed per project plan.
print('Google Drive Export has been bypassed per project plan. All results are saved locally.')


Google Drive Export has been bypassed per project plan. All results are saved locally.
